In [2]:
from google.colab import files

uploaded = files.upload()

Saving gateway_export.csv to gateway_export.csv
Saving ledger.csv to ledger.csv


In [3]:
import pandas as pd

ledger_df = pd.read_csv("ledger.csv")
gateway_df = pd.read_csv("gateway_export.csv")

print("Ledger rows:", len(ledger_df))
print("Gateway rows:", len(gateway_df))

print("\nLedger columns:")
print(ledger_df.columns.tolist())

print("\nGateway columns:")
print(gateway_df.columns.tolist())

Ledger rows: 547
Gateway rows: 530

Ledger columns:
['transaction_id', 'user_id', 'merchant_id', 'transaction_time', 'amount_inr', 'payment_method', 'status', 'risk_score']

Gateway columns:
['transaction_id', 'user_id', 'merchant_id', 'transaction_time', 'amount_inr', 'payment_method', 'status', 'risk_score']


In [4]:
def reconcile_payments(ledger_df, gateway_df):
    # Make copies so the original DataFrames are not changed
    ledger = ledger_df.copy()
    gateway = gateway_df.copy()

    # Make sure transaction IDs are comparable
    ledger_ids = set(ledger["transaction_id"])
    gateway_ids = set(gateway["transaction_id"])

    # 1. Transactions present in ledger but missing from gateway
    missing_in_gateway_ids = ledger_ids - gateway_ids

    missing_in_gateway = ledger[
        ledger["transaction_id"].isin(missing_in_gateway_ids)
    ].copy()

    # 2. Transactions present in gateway but missing from ledger
    missing_in_ledger_ids = gateway_ids - ledger_ids

    missing_in_ledger = gateway[
        gateway["transaction_id"].isin(missing_in_ledger_ids)
    ].copy()

    # 3 & 4. Compare matching transactions using merge
    matched = pd.merge(
        ledger,
        gateway,
        on="transaction_id",
        how="inner",
        suffixes=("_ledger", "_gateway")
    )

    # 3. Amount mismatches
    matched["amount_difference"] = (
        matched["amount_inr_ledger"] -
        matched["amount_inr_gateway"]
    )

    amount_mismatches = matched[
        matched["amount_difference"] != 0
    ].copy()

    # 4. Status mismatches
    status_mismatches = matched[
        matched["status_ledger"] != matched["status_gateway"]
    ].copy()

    return (
        missing_in_gateway,
        missing_in_ledger,
        amount_mismatches,
        status_mismatches
    )

In [5]:
missing_in_gateway, missing_in_ledger, amount_mismatches, status_mismatches = reconcile_payments(
    ledger_df,
    gateway_df
)

print("Transactions missing in gateway:", len(missing_in_gateway))
print("Transactions missing in ledger:", len(missing_in_ledger))
print("Amount mismatches:", len(amount_mismatches))
print("Status mismatches:", len(status_mismatches))

Transactions missing in gateway: 27
Transactions missing in ledger: 10
Amount mismatches: 16
Status mismatches: 9


In [6]:
print("=== TRANSACTIONS MISSING IN GATEWAY ===")
display(missing_in_gateway)

print("=== TRANSACTIONS MISSING IN LEDGER ===")
display(missing_in_ledger)

print("=== AMOUNT MISMATCHES ===")
display(
    amount_mismatches[
        [
            "transaction_id",
            "amount_inr_ledger",
            "amount_inr_gateway",
            "amount_difference"
        ]
    ]
)

print("=== STATUS MISMATCHES ===")
display(
    status_mismatches[
        [
            "transaction_id",
            "status_ledger",
            "status_gateway"
        ]
    ]
)

=== TRANSACTIONS MISSING IN GATEWAY ===


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
0,TXN100000,350,16,2026-01-29 11:13:00,2999,Wallet,captured,85
5,TXN100005,196,3,2026-01-06 22:06:00,99,UPI,captured,60
36,TXN100036,321,16,2026-01-25 16:31:00,49,UPI,captured,35
44,TXN100044,309,15,2026-01-24 23:34:00,99,Netbanking,captured,62
56,TXN100056,132,24,2026-01-07 14:44:00,299,Wallet,captured,21
74,TXN100074,244,32,2026-01-18 07:44:00,149,UPI,captured,57
92,TXN100092,157,37,2026-01-28 18:27:00,499,Wallet,captured,79
130,TXN100130,211,3,2026-01-13 07:54:00,149,UPI,chargeback,40
134,TXN100134,74,27,2026-01-17 14:00:00,799,UPI,captured,83
188,TXN100188,32,31,2026-01-15 20:01:00,299,UPI,captured,51


=== TRANSACTIONS MISSING IN LEDGER ===


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
520,TXNX9000,262,25,2026-01-14 00:00:00,149,Netbanking,captured,54
521,TXNX9001,96,32,2026-01-21 00:00:00,4999,Wallet,captured,71
522,TXNX9002,86,32,2026-01-10 00:00:00,149,Wallet,captured,40
523,TXNX9003,231,40,2026-01-02 00:00:00,799,UPI,captured,62
524,TXNX9004,70,13,2026-01-27 00:00:00,1499,Netbanking,captured,52
525,TXNX9005,252,27,2026-01-23 00:00:00,2999,Netbanking,captured,21
526,TXNX9006,43,37,2026-01-01 00:00:00,299,Card,captured,4
527,TXNX9007,141,15,2026-01-18 00:00:00,499,Wallet,captured,99
528,TXNX9008,235,37,2026-01-24 00:00:00,2999,UPI,captured,73
529,TXNX9009,59,18,2026-01-25 00:00:00,4999,Card,captured,69


=== AMOUNT MISMATCHES ===


,transaction_id,amount_inr_ledger,amount_inr_gateway,amount_difference
9,TXN100011,2999,2949,50
17,TXN100019,49,99,-50
36,TXN100039,1499,1399,100
194,TXN100204,49,-51,100
207,TXN100218,49,-1,50
240,TXN100252,299,349,-50
257,TXN100270,799,899,-100
265,TXN100278,149,99,50
271,TXN100284,1499,1399,100
273,TXN100286,49,149,-100


=== STATUS MISMATCHES ===


,transaction_id,status_ledger,status_gateway
39,TXN100042,captured,failed
204,TXN100215,captured,failed
254,TXN100267,captured,failed
286,TXN100300,captured,failed
371,TXN100392,captured,failed
392,TXN100414,captured,failed
463,TXN100487,captured,failed
482,TXN200008,chargeback,failed
516,TXN300027,captured,failed


In [7]:
print("=" * 50)
print("        PART C - RECONCILIATION VALIDATION")
print("=" * 50)

print(f"Ledger rows: {len(ledger_df)}")
print(f"Gateway rows: {len(gateway_df)}")

print("\n--- DISCREPANCY COUNTS ---")
print(f"Missing in gateway : {len(missing_in_gateway)}")
print(f"Missing in ledger  : {len(missing_in_ledger)}")
print(f"Amount mismatches  : {len(amount_mismatches)}")
print(f"Status mismatches  : {len(status_mismatches)}")

print("\n--- FUNCTION VALIDATION ---")

assert len(missing_in_gateway) == 27
assert len(missing_in_ledger) == 10
assert len(amount_mismatches) == 16
assert len(status_mismatches) == 9

print("PASS: All four discrepancy outputs generated successfully.")
print("PASS: Amount differences were computed.")
print("PASS: Status differences were identified.")
print("PASS: Part C reconciliation completed.")

print("\n" + "=" * 50)
print("          PART C VALIDATION COMPLETE")
print("=" * 50)

        PART C - RECONCILIATION VALIDATION
Ledger rows: 547
Gateway rows: 530

--- DISCREPANCY COUNTS ---
Missing in gateway : 27
Missing in ledger  : 10
Amount mismatches  : 16
Status mismatches  : 9

--- FUNCTION VALIDATION ---
PASS: All four discrepancy outputs generated successfully.
PASS: Amount differences were computed.
PASS: Status differences were identified.
PASS: Part C reconciliation completed.

          PART C VALIDATION COMPLETE
